# tinyLLM

This tinyLLM is based on the ***Bi Gram prediction model***. It predicts the probability of a word or a character based on the ***frequency of occurrence of the preceding word or character pair***. It analyzes text in pairs of consecutive words/characters, called bi-grams. 

The model estimates:

$P(w_n \mid w_{n-1})$

## Step 1. Create Tokens
Creating *tokens* involves spliting the corpus in to unique words and assigning them a unique ***Token ID***

- Clean the corpus

In [1]:
import re

text = """hello hello hello hello world, 
this is a tiny tiny tiny tiny tiny llm
this is a tiny tiny tiny tiny tiny llm
actually a very tiny llm
this is only for education purposes.
hope hope this helps you understand how a transformer works.
"""
# Remove newlines and extra spaces to create a single line of text for training.
# Repeat 200 times to give the model more data to learn from, 
# which will help it learn the patterns better.
text = re.sub(r'\s*[\r\n]+\s*', ' ', text).strip() * 200


### About the Corpus
- tiny - 11 times, 
- this - 4 times, 
- hello - 4 times, 
- hope - 2 times,
- helps - 1 time, 
- how - 1 time, 

#### Note that :
- 'i' follows 't' more frequently (11) than 'h' (4), so the model will learn to predict 'i' after 't' more often
- 'e' follows 'h' more frequently (5) than 'o' (3), so the model will learn to predict 'e' after 'h' more often

#### Try these char inputs:
- 't' - expected output 'i',  'h', 'r'
- 'h'  - expected output 'e', 'o', 'i', 'a'

#### Try these sentence inputs:
- 'hope t' - output still be 'i',  'h', 'r'     - whereas the sentence is 'hope this' 
- 'hope h' - output still be 'e', 'o', 'i', 'a' - whereas the sentence is 'hope hope'

- The Bi-Gram model only looks at the immediately preceding character and does not have the ability to understand the broader context, long-range dependencies or the semantic meaning of the sentence
- As a result, it predicts the next character based solely on the last character seen 


- Create *Token Index*

In [2]:
chars = sorted(list(set(text)))

vocab_size = len(chars)
print("vocab size:", vocab_size)
print(f"All the Unique characters: {chars}\n")
#print("all the unique characters:", ''.join(chars))


vocab size: 23
All the Unique characters: [' ', ',', '.', 'a', 'c', 'd', 'e', 'f', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y']



- Create a *mapping* from characters to integers

In [3]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string


- Let's test our encoder and decoder for some characters

In [4]:
text_to_encode = "hii there"
encoded_text = encode(text_to_encode)

print("Text to encode: \n"+ str(text_to_encode))
print("\nEncoded text: \n"+ str(encoded_text))
print("\nDecoded text: \n"+ str(decode(encoded_text)))    

Text to encode: 
hii there

Encoded text: 
[8, 9, 9, 0, 18, 8, 6, 16, 6]

Decoded text: 
hii there


## The BiGram Language Model

A bigram language model predicts the next token based only on the immediately preceding token, using the frequency of token pairs (bigrams)

### Important Functions

#### forward(): The Forward Pass. The model's "thinking or learning process." 
- It takes in a sequence of characters (idx) and optionally the correct next characters 
- (targets for training).
- If targets are provided, it calculates the loss; otherwise, it just returns the logits 
- for the next character predictions.

#### F.cross_entropy(): Calculates the loss between the model's predictions and the true targets.
- Higher Entropy means - you know little and cannot predict accurately
- Lower Entropy means - you know everything and can predict accurately
- Cross-Entropy compares the model's guess (logits) to the correct answer (targets) and gives a score (loss) that tells the model how well it did. 
- The model then uses this score to adjust its "brain" (parameters) to do better next time.

#### loss.backward(): The back propogation
- Which weights caused this error
- Backward pass computes which direction and how steeply the loss would decrease if you nudged each weight
- For every parameter in the network. It propagates errors backward through the network

#### optimizer()
- Optimizer use the gradients calculated in back propogation and decides how exactly to update each weight to minimize loss
- Commonly used optimizer is AdamW - Adaptive Moment Estimation and Weight Decay

#### The generate method: The model's "imagination or inference engine."
- It takes a starting point (context) and generates new characters based on what it has learned.

#### logits.shape: The tensor shape:
- B: Batch size (number of sequences processed together) - training cycle number
- T: Sequence length (number of tokens in each sequence) - context window
- C: Number of possible next characters (vocab_size) - predicted next character scores

- For every single position in the B-batch and T-time, there are C=vocab_size scores predicting what comes next
- This dimension C holds the logits (raw prediction scores) for every possible next character in the vocabulary



In [5]:
import torch
import torch.nn as nn
from torch.nn import functional as F

##############################################################################
## The "Brain" (BiGram Model)
##############################################################################


import torch.nn as nn
from torch.nn import functional as F

class BiGramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Create a lookup table (embedding layer) that maps each token index to a vector of size vocab_size.
        # Initially this embedding is random, but during training, it will learn to represent the relationships between characters.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    # This is the forward-pass Computation, which is the model's core prediction step
    # Training: It takes in a sequence of token indices (idx) 
    # and the corresponding target indices (targets) and compute cross-entropy loss
    # Inference: Returns the logits for the next character predictions.
    # This function is implicitly called when the model instance is called with input data.
    # During training with the targets provided model(xb, yb), it computes the loss. 
    # During inference without targets self(idx) called from generate, it just returns the logits.
    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) 
        ## The shape of idx is (B, T) where B is the batch size and T is the sequence length.
        ## The token_embedding_table is a lookup table that maps each token index to a vector of size vocab_size.
        ## Therefore, for each (B, T) it will return a vector of size (C), of size vocab_size.
        ## This is the prediction step where the model looks at the 
        ## current token and produces a score for each possible next token in the vocabulary.
        ## resulting in a tensor of shape (B, T, C)
        #### Note: In the initial run with a random state, these logits are just random scores, 
        ####       but as the model trains, it will learn to produce more meaningful scores that reflect
        ####       the likelihood of each next character based on the training data. 

        if targets is None:
            # This is the inference mode, where we just want to get the logits for the next character predictions.
            loss = None
        else:
            #This is the training mode, where we compute the loss by comparing the predicted logits to the target indices.
            B, T, C = logits.shape
            ## cross_entropy expects a 2D input
            ## where each row is an example and each column is a score for a category
            ## logits = logits.view(B*T, C) - flattens 3D tensor to 2D by combining batch and time dimensions
            ## targets = targets.view(B*T) - flattens targets to a 1D tensor of the same length as the number of examples in logits
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            ## Note: Since the target the occurance of a character following another character 
            ## is not deterministic, the model will learn to assign probabilities to the next characters 
            ## based on their frequency in the training data.
            ## This is where the learning and adjusting the parameter weights is happening based on the loss calculated from the model's predictions and the actual targets.
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        # Use the forward method in the inference mode to get the logits for the next character predictions,
        # The logits returned contains the scores for all possible next characters in the vocabulary 
        # for each position in the input sequence.
        # The softmax function is applied to these logits to convert them into probabilities, 
        # which represent the likelihood of each possible next character.
        for _ in range(max_new_tokens):
            # Get the logits for the next character predictions based on the current context (idx).
            logits, loss = self(idx)

            #take only the predictions corresponding to the last token in the current context,
            logits = logits[:, -1, :] # focus only on the last time step

            # Converts raw scores into probabilities for the next character prediction.
            probs = F.softmax(logits, dim=-1) # get probabilities
            
            # Pick the next token index using the model’s probability distribution
            idx_predict = torch.multinomial(probs, num_samples=1)

            # Append the sampled index to the current context (idx) to form the new context for the next prediction. 
            idx = torch.cat((idx, idx_predict), dim=1) # append sampled index
        return idx


## Step 2: Training the Model

#### Step 2a: Define hyper-parameters for the training

In [6]:
# ----- Training Parameters -----
learning_rate = 1e-3 #Default - 3e-4

torch.manual_seed(1337)

# ----- Check for GPU CUDA -----
#print(f"Is CUDA available? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device Available. Name: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA device found. Training will be done on CPU.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#print(f"Using device: {device}")

# Force CPU for more precise embedding values (comment out if you want to use GPU for faster training)
device = "cpu"


CUDA Device Available. Name: NVIDIA RTX 500 Ada Generation Laptop GPU


In [7]:
# Setup the no of training iterations and evaluation interval based on user input
try:
    max_iters =  int(input("Enter no of iterations: "))
    if max_iters <= 0:
        print("Invalid input for iterations. using default 100.")
        max_iters = 100
except ValueError:
    print("Invalid input for iterations. using default 100.")
    max_iters = 100
if max_iters > 1000:
    eval_interval = 200
elif max_iters > 20:
    eval_interval = max_iters // 10
else:    eval_interval = 1

print(f"Training for {max_iters} iterations with evaluation every {eval_interval} iterations.")

Training for 1 iterations with evaluation every 1 iterations.


#### Step 2b: Create a ***Tensor*** to hold the corpus

In [8]:
# Create a tensor holding the encoded version of the text
data = torch.tensor(encode(text), dtype=torch.long)
#print(data.shape, data.dtype)
print(data[:100])

tensor([ 8,  6, 11, 11, 14,  0,  8,  6, 11, 11, 14,  0,  8,  6, 11, 11, 14,  0,
         8,  6, 11, 11, 14,  0, 21, 14, 16, 11,  5,  1,  0, 18,  8,  9, 17,  0,
         9, 17,  0,  3,  0, 18,  9, 13, 22,  0, 18,  9, 13, 22,  0, 18,  9, 13,
        22,  0, 18,  9, 13, 22,  0, 18,  9, 13, 22,  0, 11, 11, 12,  0, 18,  8,
         9, 17,  0,  9, 17,  0,  3,  0, 18,  9, 13, 22,  0, 18,  9, 13, 22,  0,
        18,  9, 13, 22,  0, 18,  9, 13, 22,  0])


#### Step 2c: Execute the training iterations

In [9]:
import time
## Training the Model
## We show the model the text over and over again, 
## and it "learns" which letters usually follow each other 
## (like 'h' followed by 'e').


model = BiGramLanguageModel(vocab_size).to(device)
print(f"No. of Parameters: {sum(p.numel() for p in model.parameters())}")
print("*Note: In this model the number of parameters is equal to vocab_size^2 because each token has a vector of size vocab_size, and there are vocab_size tokens.")
print(f"Training on device: {device}\n")
print("-" * 70 + "\n")

# Inspect the initial embedding for 'h' before training
h_id = stoi['h'] if 'h' in stoi else None
h_emb_before_rounded = None
if h_id is not None:
    embedding_device = model.token_embedding_table.weight.device
    h_emb_before = model.token_embedding_table(
        torch.tensor([h_id], dtype=torch.long, device=embedding_device)
    ).squeeze(0)
    h_emb_before_rounded = [round(v, 3) for v in h_emb_before.tolist()]


## AdamW: A smart way to adjust the model's "brain" based on the loss.
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

i=0
h_emb_rounded = []
train_start_time = time.time()
for iters in range(max_iters): 
    # Sample a batch of data 
    #--------------------- highly simplified ------------------
    # In this case the xb and yb will be exactly the same in every training step
    xb, yb = data[:-1].unsqueeze(0), data[1:].unsqueeze(0)

    #Move the tensors to the same device as the model (GPU or CPU) for computation
    xb, yb = xb.to(device), yb.to(device)

    #--------------------- random chunks ------------------
    # In this case the xb and yb will be different slice of the data in every training step, 
    # which will help the model learn better by seeing different parts of the text.
    #### For using the simple training data comment this
    #xb, yb = get_chunks(data)
    #xb, yb = xb.to(device), yb.to(device)

    ## xb is all elements of data except the last one, 
    ## yb is all elements of data except the first one.
    ## for a sequence [1, 2, 3, 4], 
    ## xb becomes [1, 2, 3] 
    ## and yb becomes [2, 3, 4]
    ## At position 0, the input is 1 and the target is 2
    ## .unsqueeze(0) inserts a new dimension at index 0
    ## The final shape of xb and yb becomes (1, T), where B=1 and T is the length of the sequence minus one.

    # Evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)

    # Backpropagation: The model's "learning process" where it adjusts its parameters based on the loss.
    loss.backward()
    optimizer.step()

    if iters % eval_interval == 0:
        elapsed_time = time.time() - train_start_time
        print(f"iters {iters}: loss {loss.item():.4f}, elapsed time: {elapsed_time:.2f}s")
        
        # Inspect the learned embedding for 'h' after training 
        if h_id is not None:
            embedding_device = model.token_embedding_table.weight.device
            h_emb = model.token_embedding_table(
                torch.tensor([h_id], dtype=torch.long, device=embedding_device)
            ).squeeze(0)
            h_emb_rounded.append([round(v, 3) for v in h_emb.tolist()])

#embedding_weight = model.token_embedding_table.weight
#print(f"Embedding matrix shape (vocab_size x embedding_dim): {tuple(embedding_weight.shape)}\n")
print("-" * 70 + "\n")
elapsed_time = time.time() - train_start_time
print(f"Training completed in {elapsed_time:.2f}s")


No. of Parameters: 529
*Note: In this model the number of parameters is equal to vocab_size^2 because each token has a vector of size vocab_size, and there are vocab_size tokens.
Training on device: cpu

----------------------------------------------------------------------

iters 0: loss 3.6717, elapsed time: 0.02s
----------------------------------------------------------------------

Training completed in 0.02s


#### How the embeddings are changing with training

In [10]:
# Print the embedding for 'h' before and after training to see how it has changed
print(f"Initial embedding for 'h' (id={h_id}): {h_emb_before_rounded}\n")

print("How the embedding are changing during training:")
for each in h_emb_rounded:
    print(f"Learned embedding for 'h' (id={h_id}): {each}")


Initial embedding for 'h' (id=8): [0.959, 1.06, 0.63, -1.287, -0.687, 2.138, 0.511, 1.219, 0.191, -0.343, 1.796, 1.391, 1.079, -0.615, -0.459, 0.567, 0.018, -1.661, 1.117, 0.52, -1.242, -0.962, -0.085]

How the embedding are changing during training:
Learned embedding for 'h' (id=8): [0.958, 1.059, 0.629, -1.288, -0.688, 2.137, 0.512, 1.218, 0.19, -0.342, 1.794, 1.39, 1.078, -0.616, -0.458, 0.566, 0.017, -1.662, 1.116, 0.519, -1.243, -0.963, -0.086]


## Step 3: Intference stage. Generating the text

In [11]:
#context = torch.zeros((1, 1), dtype=torch.long, device=device)
#stoi[c] if c in stoi else fallback_idx for c in prompt

fallback_idx = ' '
embedding_device = model.token_embedding_table.weight.device


In [12]:

input_char = 't'  # Example input character

context_ids = [stoi[c] if c in stoi else fallback_idx for c in input_char]
print(f"Input character: '{input_char}' -> Encoded as: {context_ids}")

#context = torch.zeros((1, 1), dtype=torch.long)
context = torch.tensor([context_ids], dtype=torch.long, device=embedding_device)

context_len = context.shape[1]
#print(f"Context shape: {context.shape}, Context content (encoded): {context.tolist()}")
generated_ids = model.generate(context, max_new_tokens=50)[0].tolist()
new_generated_ids = generated_ids[context_len:]

print(f">> TinyLLM (predicted next character): {decode(new_generated_ids[0:1])}")
print(f">> TinyLLM (complete generation): {decode(generated_ids)}\n")


Input character: 't' -> Encoded as: [18]
>> TinyLLM (predicted next character): f
>> TinyLLM (complete generation): tf,hdhtomnw,iksdocni, dos uaohdy  uf,o ify.pmacskyc

